# **Prediccion GPU con pesos guardados**

Se recuperan los pesos de Drive y se ejecuta forward con los kernels CUDA propios.
No se entrena ni se modifican los archivos del modelo. Se utiliza validacion y la
particion de prueba permanece reservada. Seleccionar una GPU NVIDIA en Colab.

Se incluye una comprobacion breve CPU/GPU con pesos identicos. Las mediciones de
latencia son diagnosticas; las pruebas independientes y los experimentos repetidos
corresponden a la etapa de evaluacion del proyecto.


## **Preparar el repositorio**

Se ejecuta al iniciar el notebook. Si la carpeta existe, se actualizan las
referencias; si no existe, se clona. Se muestra el commit utilizado. Si cambian
modulos Python ya importados, se reinicia la sesion antes de continuar.


In [ ]:
from pathlib import Path
import subprocess

url_repositorio = "https://github.com/Bamo0507/Proyecto_3_Cuda_CNN.git"
revision_repositorio = "origin/main"
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

if directorio_proyecto.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=directorio_proyecto, check=True)
else:
    subprocess.run(["git", "clone", url_repositorio, str(directorio_proyecto)], check=True)

subprocess.run(
    ["git", "checkout", "--detach", revision_repositorio],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Instalar dependencias CUDA**

Se instalan las dependencias de Colab y Matplotlib. Si se solicita reiniciar la
sesion, hacerlo antes de continuar y ejecutar nuevamente la preparacion del repositorio.


In [ ]:
import subprocess
import sys

resultado_instalacion = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(directorio_proyecto / "requirements_colab.txt"), "matplotlib"],
    capture_output=True,
    text=True,
)
print(resultado_instalacion.stdout, end="")
if resultado_instalacion.stderr:
    print(resultado_instalacion.stderr, end="")
resultado_instalacion.check_returncode()


## **Montar Drive y seleccionar la corrida**

Se utiliza `gpu_01`, respaldada despues del entrenamiento. La carpeta
`proyecto_senias` debe estar en Mi unidad o tener un acceso directo con ese nombre.
Tambien pueden usarse pesos CPU: el formato de persistencia es compartido.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

directorio_drive = Path("/content/drive/MyDrive/proyecto_senias")
nombre_ejecucion = "gpu_01"


## **Copiar pesos y cargar validacion**

Se copian los pesos, el registro y el NPZ al disco temporal de Colab. Se comprueba
el orden de clases del modelo y del dataset, y se cargan solamente las imagenes
y etiquetas de validacion. Los pesos se recuperan tal como fueron guardados.


In [ ]:
import json
import shutil
import sys
import numpy as np

sys.path.insert(0, str(directorio_proyecto / "src"))

import config
from compartido.persistencia import cargar_parametros
from numba import cuda
from gpu.memoria import preparar_memoria
from gpu.prediccion import predecir

directorio_modelo_drive = directorio_drive / "resultados" / nombre_ejecucion
ruta_modelo_local = config.DIRECTORIO_MODELOS / f"{nombre_ejecucion}.npz"
ruta_registro_local = config.DIRECTORIO_RESULTADOS / f"{nombre_ejecucion}.json"

for ruta_origen, ruta_destino in (
    (directorio_modelo_drive / f"{nombre_ejecucion}.npz", ruta_modelo_local),
    (directorio_modelo_drive / f"{nombre_ejecucion}.json", ruta_registro_local),
    (directorio_drive / "02_tensores.npz", config.RUTA_TENSORES),
):
    ruta_destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(ruta_origen, ruta_destino)

registro_modelo = json.loads(ruta_registro_local.read_text())
if registro_modelo["configuracion"]["clases"] != list(config.CLASES):
    raise ValueError("El orden de clases del modelo no coincide con config.py.")
if registro_modelo["configuracion"]["forma_entrada"] != list(config.FORMA_ENTRADA):
    raise ValueError("La forma de entrada del modelo no coincide con config.py.")

parametros = cargar_parametros(ruta_modelo_local)
with np.load(config.RUTA_TENSORES, allow_pickle=False) as archivo_tensores:
    if not np.array_equal(archivo_tensores["clases"], np.asarray(config.CLASES)):
        raise ValueError("El orden de clases del dataset no coincide con el modelo.")
    imagenes_validacion = archivo_tensores["x_val"]
    etiquetas_validacion = archivo_tensores["y_val"]

if etiquetas_validacion.shape != (len(imagenes_validacion),):
    raise ValueError("Debe existir una etiqueta por imagen de validacion.")
print("Modelo cargado:", nombre_ejecucion)
print("Imagenes de validacion:", len(imagenes_validacion))


## **Comprobar integracion CPU y GPU**

Se ejecuta el mismo lote de dos imagenes con pesos identicos en ambos backends.
Primero se comparan forward, perdida y gradientes con la inicializacion compartida;
luego se compara forward y perdida con los pesos guardados de esta corrida.
No se aplica Adam ni se escriben pesos. La semilla proviene de `config.py`.

Se muestran errores absolutos maximos por capa y se exige
`abs(GPU - CPU) <= 1e-5 + 1e-3 * abs(CPU)` para cada elemento finito.
Las tolerancias son un criterio inicial de integracion, no una garantia general.
Si aparece `REVISAR`, conservar la salida para analizar la primera divergencia;
no ampliar tolerancias automaticamente. Las etiquetas se muestran aparte:
probabilidades muy cercanas pueden cambiar el argmax en un empate.

La CPU compilara sus operaciones en esta celda. Esto no es una medicion de velocidad
ni una evaluacion de todo el dataset.


In [ ]:
import subprocess
import sys

comando_comprobacion = [
    sys.executable, "-u", str(directorio_proyecto / "src/verificar_equivalencia.py"),
    "--ruta-tensores", str(config.RUTA_TENSORES),
    "--cantidad-imagenes", "2",
]
for argumentos_adicionales in (
    [],
    ["--ruta-modelo", str(ruta_modelo_local), "--solo-forward"],
):
    comando_actual = comando_comprobacion + argumentos_adicionales
    with subprocess.Popen(
        comando_actual,
        cwd=directorio_proyecto,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as proceso_comprobacion:
        for linea_salida in proceso_comprobacion.stdout:
            print(linea_salida, end="", flush=True)
        codigo_salida = proceso_comprobacion.wait()
        if codigo_salida != 0:
            raise subprocess.CalledProcessError(codigo_salida, comando_actual)


## **Preparar memoria y realizar la primera prediccion**

Se copian los pesos a GPU y se reservan buffers una sola vez. La primera prediccion
puede incluir compilacion y se registra por separado. La copia del resultado a RAM
completa los kernels pendientes antes de devolver la etiqueta.


In [ ]:
import time

if cuda.config.ENABLE_CUDASIM or not cuda.is_available():
    raise RuntimeError("Selecciona una GPU NVIDIA real antes de continuar.")
print("GPU:", cuda.get_current_device().name)

inicio_preparacion = time.perf_counter()
memoria_prediccion = preparar_memoria(
    parametros, tamano_lote=config.TAMANO_LOTE, entrenamiento=False
)
cuda.synchronize()
tiempo_preparacion = time.perf_counter() - inicio_preparacion

indice_imagen = 0
imagen_individual = imagenes_validacion[indice_imagen:indice_imagen + 1]
cuda.synchronize()
inicio_calentamiento = time.perf_counter()
prediccion_individual = predecir(imagen_individual, memoria_prediccion)[0]
tiempo_primera_prediccion = time.perf_counter() - inicio_calentamiento

print("Etiqueta real:", config.CLASES[int(etiquetas_validacion[indice_imagen])])
print("Prediccion:", prediccion_individual["etiqueta"])
print(f"Probabilidad: {prediccion_individual['probabilidad']:.2%}")
print(f"Preparacion de memoria y pesos: {tiempo_preparacion:.4f} s")
print(f"Primera prediccion y posible compilacion: {tiempo_primera_prediccion:.4f} s")


## **Revisar ejemplos de validacion**

Se seleccionan cinco ejemplos con semilla fija y se muestran sus etiquetas reales
y predichas. El color permite distinguir aciertos y errores; estos ejemplos no
sustituyen la accuracy calculada sobre toda la validacion.


In [ ]:
import matplotlib.pyplot as graficos

cantidad_ejemplos = min(5, len(imagenes_validacion))
generador_aleatorio = np.random.default_rng(config.SEMILLA_ALEATORIA)
indices_ejemplos = generador_aleatorio.choice(
    len(imagenes_validacion), size=cantidad_ejemplos, replace=False
)
predicciones_ejemplos = predecir(imagenes_validacion[indices_ejemplos], memoria_prediccion)

figura, ejes = graficos.subplots(
    1, cantidad_ejemplos, figsize=(3 * cantidad_ejemplos, 3), squeeze=False
)
for indice_ejemplo, indice_validacion in enumerate(indices_ejemplos):
    prediccion = predicciones_ejemplos[indice_ejemplo]
    etiqueta_real = config.CLASES[int(etiquetas_validacion[indice_validacion])]
    eje_imagen = ejes[0, indice_ejemplo]
    eje_imagen.imshow(imagenes_validacion[indice_validacion, :, :, 0], cmap="gray", vmin=0, vmax=1)
    eje_imagen.set_title(
        f"Real: {etiqueta_real}\nPred: {prediccion['etiqueta']} ({prediccion['probabilidad']:.1%})",
        color="green" if prediccion["etiqueta"] == etiqueta_real else "red",
    )
    eje_imagen.axis("off")
graficos.tight_layout()
graficos.show()


## **Medir la prediccion despues del calentamiento**

Se repite 30 veces la prediccion de una imagen. Cada tiempo incluye transferencia
de la imagen RAM a GPU, forward, recuperacion de probabilidades y seleccion de
etiqueta en CPU. Los pesos y buffers permanecen en GPU. Se excluyen Drive, carga
del modelo, reserva inicial y compilacion.

Esta comprobacion de latencia no sustituye los experimentos de Daniela. Para
comparar backends, utilizar los mismos pesos, entradas y condiciones.


In [ ]:
cantidad_repeticiones = 30
tiempos_prediccion = []

for numero_repeticion in range(cantidad_repeticiones):
    cuda.synchronize()
    inicio_prediccion = time.perf_counter()
    prediccion_medida = predecir(imagen_individual, memoria_prediccion)[0]
    tiempos_prediccion.append(time.perf_counter() - inicio_prediccion)

tiempos_milisegundos = np.asarray(tiempos_prediccion) * 1000
print("Repeticiones:", cantidad_repeticiones)
print(f"Latencia promedio: {tiempos_milisegundos.mean():.4f} ms")
print(f"Desviacion estandar: {tiempos_milisegundos.std(ddof=1):.4f} ms")
print("Tiempos individuales (ms):", tiempos_milisegundos.tolist())
